In [1]:
# Import des librairies pour l'EDA de Kayak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [2]:
final_df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA /Projets_certif/02_kayak/data/final_data_cleaned.csv")

In [3]:
pd.set_option('display.max_columns', None)
final_df.head(2)

,city,lat,lon,visibility,dt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,wind.deg,wind.gust,clouds.all,sys.sunrise,sys.sunset,rain.1h,weather.0.id,weather.0.main,weather.0.description,weather.0.icon,hotel_city,hotel_name,price,latitude,longitude,rate,rate_label,experience
0,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Beauvoir,La Villa du Manoir,€ 521,48.596052,-1.500324,"9,8",Exceptionnel,6
1,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Huisnes-sur-Mer,Escale du Mont,€ 364,48.621474,-1.447158,"7,5",Bien,2


In [4]:
df = final_df.copy()

In [5]:
if "rain.1h" not in df.columns:
    df["rain.1h"] = 0

df["rain.1h"] = df["rain.1h"].fillna(0)

In [6]:
if df["wind.gust"].isnull().any():
    df["wind.gust"] = df["wind.gust"].fillna(0)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875 entries, 0 to 874
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   city                   875 non-null    object 
 1   lat                    875 non-null    float64
 2   lon                    875 non-null    float64
 3   visibility             875 non-null    int64  
 4   dt                     875 non-null    int64  
 5   main.temp              875 non-null    float64
 6   main.feels_like        875 non-null    float64
 7   main.temp_min          875 non-null    float64
 8   main.temp_max          875 non-null    float64
 9   main.pressure          875 non-null    int64  
 10  main.humidity          875 non-null    int64  
 11  main.sea_level         875 non-null    int64  
 12  main.grnd_level        875 non-null    int64  
 13  wind.speed             875 non-null    float64
 14  wind.deg               875 non-null    int64  
 15  wind.g

In [8]:
df.columns

Index(['city', 'lat', 'lon', 'visibility', 'dt', 'main.temp',
       'main.feels_like', 'main.temp_min', 'main.temp_max', 'main.pressure',
       'main.humidity', 'main.sea_level', 'main.grnd_level', 'wind.speed',
       'wind.deg', 'wind.gust', 'clouds.all', 'sys.sunrise', 'sys.sunset',
       'rain.1h', 'weather.0.id', 'weather.0.main', 'weather.0.description',
       'weather.0.icon', 'hotel_city', 'hotel_name', 'price', 'latitude',
       'longitude', 'rate', 'rate_label', 'experience'],
      dtype='object')

In [11]:
########################
# EDA pour estimer les pondérations des critères de notation des hôtels
########################

Définition des critères de sélection des hôtels pour l'analyse du top 20 des hotels en fonction des paramètres météo et des avis clients :

- Critères météorologiques : cf ==> top_5_villes.ipynb
  - Température ressentie
  - Précipitations à 1h
  - Rafales de vent
  - Vitesse du vent
  - couverture nuageuse

- Critères des avis clients :
  - Note moyenne
  - Nombre d'avis
  - Distance par rapport au centre-ville

In [12]:
eda_df = df.copy()

In [13]:
eda_df = eda_df[["city", "hotel_city", "latitude", "longitude", "hotel_name", "rate", "experience","price"]]

In [ ]:
# Distribution de la note moyenne des hôtels en fonction du nombre d'avis clients. 
dis_rate_exp_fig = px.(eda_df, x="experience", y="rate", color="city", hover_data=["hotel_name", "price"], title="Distribution de la note moyenne des hôtels en fonction du nombre d'avis clients")

In [ ]:
dis_rate_exp_fig.update_layout(
    xaxis_title="Nombre d'avis clients",
    yaxis_title="Note moyenne des hôtels",
    legend_title="Ville",
    font=dict(
        family="Arial",
        size=12,
        color="black"
    )
)


Le top 20 des hôtels sera sélectionné en fonction d'une combinaison pondérée de ces critères, en donnant plus de poids aux avis clients pour refléter l'expérience globale des visiteurs.

Critères de notation pour la météo : # fonction weather_scores(df_cws)
Liste déterministe pour la notation des villes en fonction des conditions météorologiques, où chaque critère est évalué et pondéré pour obtenir un score global de confort météorologique.

Critères de notation pour les avis clients : # fonction review_scores(df_hotels)
Liste déterministe pour la notation des hôtels en fonction des avis clients, où chaque critère est évalué et pondéré pour obtenir un score global de satisfaction des clients.

#### En étude ####

Pondération des critères de notation pour les hôtels :
- Note moyenne : 0.4
- Nombre d'avis : 0.3
- Distance par rapport au centre-ville : 0.3

liste deterministre des critères de notation : 
    
    - Note moyenne :
        - 5 étoiles : 1.0
        - 4 étoiles : 0.8
        - 3 étoiles : 0.6
        - 2 étoiles : 0.4
        - 1 étoile : 0.2
        - pas d'avis : 0.0
    
    - Nombre d'avis :
        - Plus de 100 avis : 1.0
        - Entre 50 et 100 avis : 0.8
        - Entre 20 et 50 avis : 0.6
        - Entre 10 et 20 avis : 0.4
        - Moins de 10 avis : 0.2
        - pas d'avis : 0.0

    - Distance par rapport au centre-ville :
        - Moins de 1 km : 1.0
        - Entre 1 et 3 km : 0.8
        - Entre 3 et 5 km : 0.6
        - Entre 5 et 10 km : 0.4
        - Plus de 10 km : 0.2
        - pas d'information : 0.0

In [19]:
df_hotels = df[["city", "hotel_city", "latitude", "longitude", "hotel_name", "rate", "experience","price"]]

In [20]:
df_hotels.head(2)

,city,hotel_city,latitude,longitude,hotel_name,rate,experience,price
0,mont saint michel,Beauvoir,48.596052,-1.500324,La Villa du Manoir,"9,8",6,€ 521
1,mont saint michel,Huisnes-sur-Mer,48.621474,-1.447158,Escale du Mont,"7,5",2,€ 364
